<a href="https://colab.research.google.com/github/callealbrecht/Constructing-an-algorithm/blob/main/Constructing_an_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Problem setup

We have:
- N robots on a spacecraft.
- Exactly K of them are corrupted.
- An uncorrupted robot always tells the truth when testing another robot.
- A corrupted robot can say anything it wants, and corrupted robots can coordinate.

There is a single test bed that can load two robots at a time.

Each robot reports whether the other robot is corrupted.



In [ ]:
import random
from typing import List, Tuple, Optional, Set


In [ ]:
class Robot:
    """
    Robot on the spacecraft.
    - If uncorrupted: always tells the truth.
    - If corrupted: may lie. Here we give them a simple adversarial strategy:
      they always say that the other robot is 'good', to hide corruption.
    """
    def __init__(self, robot_id: int, is_corrupted: bool):
        self.id = robot_id
        self.is_corrupted = is_corrupted

    def assess(self, other: "Robot") -> str:
        """
        Return 'good' or 'corrupted' as assessment of 'other'.
        """
        if not self.is_corrupted:
            # truthful behaviour
            return "corrupted" if other.is_corrupted else "good"
        else:
            # adversarial behaviour (for Problem 2): always claim 'good'
            return "good"


def test_bed(i: int, j: int, robots: List[Robot]) -> Tuple[str, str]:
    """
    Put robots i and j on the test bed.
    Returns (ri, rj) where:
        ri = robot i's opinion of robot j
        rj = robot j's opinion of robot i
    """
    ri = robots[i].assess(robots[j])
    rj = robots[j].assess(robots[i])
    return ri, rj


## Phase 1: Finding a definitely uncorrupted robot

Used an elimination algorithm:

1. Start with all robots in a set S.
2. Pair them up and test each pair once.
3. For each pair (a, b):
   - If both say "good" about each other, keep one of them in the next round.
   - If there is any accusation, discard the whole pair.
4. If there is an unpaired leftover robot, carry it to the next round.
5. Repeat until only one robot is left.  

Assuming uncorrupted robots are in the *majority* (`K < N/2`), the final survivor is guaranteed to be uncorrupted.


In [ ]:
def find_definitely_good_robot(robots: List[Robot]) -> Tuple[int, int]:
    """
    Run the elimination algorithm to find one definitely uncorrupted robot.

    Returns:
        good_index: index of a robot that is guaranteed uncorrupted
        tests_used: number of test-bed uses in this phase
    """
    S = list(range(len(robots)))  # candidate indices
    tests_used = 0

    while len(S) > 1:
        new_S = []
        i = 0
        # Process pairs
        while i + 1 < len(S):
            a, b = S[i], S[i + 1]
            ra, rb = test_bed(a, b, robots)
            tests_used += 1

            if ra == "good" and rb == "good":
                # Could be (good, good) or (bad, bad); keep one representative
                new_S.append(a)
            # Otherwise: some accusation => drop both

            i += 2

        # If one leftover, carry it to next round
        if i < len(S):
            new_S.append(S[i])

        S = new_S

    good_index = S[0]
    return good_index, tests_used


## Phase 2: Classifying all robots using the good one

Once we have a robot g that we know is uncorrupted:

1. Test g with every other robot r.
2. Trust only what g says.
3. If g says "r is corrupted", then r is corrupted.
4. Otherwise, r is uncorrupted.

In [ ]:
def classify_all_robots(robots: List[Robot], good_index: int) -> Tuple[List[str], int]:
    """
    Use the known good robot to classify all robots.

    Returns:
        labels: list of "good" / "corrupted" for each robot
        tests_used: number of tests in this phase
    """
    n = len(robots)
    labels = ["unknown"] * n
    labels[good_index] = "good"
    tests_used = 0

    for i in range(n):
        if i == good_index:
            continue
        ri, _ = test_bed(good_index, i, robots)
        tests_used += 1
        # ri is truthful because good_index is uncorrupted
        labels[i] = ri

    return labels, tests_used


## Simulation:

Now:

- Create N robots with exactly K corrupted ones.
- Run Phase 1 to find a definitely good robot.
- Run Phase 2 to classify everyone.
- Compare the detected corrupted robots to the ground truth.


In [ ]:
def generate_robots(n: int, k: int, seed: Optional[int] = None) -> Tuple[List[Robot], Set[int]]:
    """
    Generate n robots with exactly k corrupted ones.
    Returns (robots_list, corrupted_indices).
    """
    assert 0 <= k <= n
    if seed is not None:
        random.seed(seed)

    indices = list(range(n))
    corrupted_indices = set(random.sample(indices, k))
    robots = [Robot(i, i in corrupted_indices) for i in indices]
    return robots, corrupted_indices


# Demo run
N = 15      # total robots
K = 5       # corrupted robots (must satisfy K < N/2 for algorithm to work)

robots, true_corrupted = generate_robots(N, K, seed=42)

print(f"Total robots: {N}")
print(f"True corrupted robots: {sorted(true_corrupted)}")

# Phase 1
good_idx, tests_phase1 = find_definitely_good_robot(robots)
print(f"\nPhase 1: found definitely good robot: {good_idx}")
print(f"Phase 1 tests used: {tests_phase1}")

# Phase 2
labels, tests_phase2 = classify_all_robots(robots, good_idx)
print("\nPhase 2: classification of all robots:")
for i, lab in enumerate(labels):
    print(f"  Robot {i}: {lab}")

print(f"\nPhase 2 tests used: {tests_phase2}")
print(f"Total tests used: {tests_phase1 + tests_phase2}")

detected_corrupted = {i for i, lab in enumerate(labels) if lab == "corrupted"}
print(f"\nDetected corrupted robots: {sorted(detected_corrupted)}")
print("Detection correct?", detected_corrupted == true_corrupted)


Total robots: 15
True corrupted robots: [0, 1, 4, 10, 11]

Phase 1: found definitely good robot: 6
Phase 1 tests used: 11

Phase 2: classification of all robots:
  Robot 0: corrupted
  Robot 1: corrupted
  Robot 2: good
  Robot 3: good
  Robot 4: corrupted
  Robot 5: good
  Robot 6: good
  Robot 7: good
  Robot 8: good
  Robot 9: good
  Robot 10: corrupted
  Robot 11: corrupted
  Robot 12: good
  Robot 13: good
  Robot 14: good

Phase 2 tests used: 14
Total tests used: 25

Detected corrupted robots: [0, 1, 4, 10, 11]
Detection correct? True


## Problem 2: Strategy for the corrupted robots

In the simulation, corrupted robots follow a simple adversarial strategy:

- **They always say that the other robot is good.**

This leads to:

- Hide from detection as long as possible.
- Make pairs of corrupted robots look the same as pairs of good robots.
- Force any correct algorithm to spend many tests before it can be confident.

## Problem 3:

It becomes impossible to guarantee correct identification when the corrupted robots are at least half of all robots(`K < N/2`). Because they can coordinate lies to perfectly imitate a different configuration of corrupted/good robots, making the observations indistinguishable to any algorithm.